# WebLooper - Lyrics Processor (Google Colab)

**This notebook runs the heavy AI (lyrics transcription + timing) using your free Colab GPU and writes results back to your Google Drive.**

It integrates perfectly with weblooper's existing Drive sync.

### Quick Steps
1. Runtime → Change runtime type → GPU (T4 recommended, free).
2. Run all cells (it will ask for Drive permission).
3. In the config cell, the `SESSION_FOLDER_ID` is usually already filled (when you clicked 'Run in my Colab' from weblooper it uploaded a ready copy with the ID baked in). If empty, paste the folder ID that weblooper shows you.
4. Cell 1 mounts Drive and installs dependencies (pip download cache is stored on Drive so subsequent runs are much faster \u2014 ~1-2 min vs ~5 min). On first run it **automatically restarts the runtime**. This is expected! After restart, re-run from Cell 2 onward (or just \"Run all\" again \u2014 Cell 1 will detect packages are already installed and skip).
5. In the config cell choose your model (USE_PARAKEET recommended as the best for singing; exactly one must be True). The notebook will find your `vocals.webm`, run the chosen model, and write `lyricTrack.json` + patch `meta.json`.
6. Go back to weblooper and click 'Load results from Colab/Drive' (or reload) — lyrics appear with proper timing!

In [ ]:
# @title 1. Install dependencies (persisted to Drive)
import subprocess, sys, os, shutil
from google.colab import drive

# --- Mount Drive early ---
drive.mount('/content/drive', force_remount=False)

# 1. Define the persistent installation directory on Drive
ENV_PATH = '/content/drive/MyDrive/.weblooper_colab_env'
os.makedirs(ENV_PATH, exist_ok=True)

# 2. Add Drive path AFTER system paths so Colab's pre-installed packages
#    (torch, torchvision, torchaudio, numpy) always take priority.
#    Drive only fills in what's missing (nemo, whisperx, pydub, etc.).
if ENV_PATH not in sys.path:
    sys.path.append(ENV_PATH)

os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{ENV_PATH}"

def _python_deps_installed():
    """Check if our heavy Python packages are readable from Drive."""
    try:
        import nemo.collections.asr
        import pydub
        return True
    except (ImportError, AttributeError):
        return False

def _system_deps_installed():
    """Check if system packages are installed on the local Colab disk."""
    return shutil.which('ffmpeg') is not None and shutil.which('sox') is not None

def _cleanup_torch_from_target():
    """Remove torch/torchvision/torchaudio from Drive target to avoid conflicts.
    Colab's pre-installed versions (compiled for this runtime's CUDA) must be used."""
    removed = []
    for pkg in ['torch', 'torchvision', 'torchaudio', 'nvidia', 'triton']:
        pkg_path = os.path.join(ENV_PATH, pkg)
        if os.path.exists(pkg_path):
            shutil.rmtree(pkg_path)
            removed.append(pkg)
        # Also remove .dist-info directories
        for d in os.listdir(ENV_PATH):
            if d.startswith(pkg) and '.dist-info' in d:
                shutil.rmtree(os.path.join(ENV_PATH, d))
    if removed:
        print(f'  Cleaned conflicting packages from Drive: {removed}')

# --- Step 1: Fast System Dependencies (~10 seconds) ---
if not _system_deps_installed():
    print('Installing system dependencies (ffmpeg, sox, libsndfile1)...')
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'sox', 'libsndfile1', 'ffmpeg'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('System deps installed  \u2713')
else:
    print('System deps already present  \u2713')

# --- Step 2: Clean up any torch packages that leaked into Drive target ---
_cleanup_torch_from_target()

# --- Step 3: Persistent Python Dependencies ---
if _python_deps_installed():
    print('Python dependencies found on Google Drive! Skipping all pip installs \u2728')
else:
    print(f'Installing Python packages directly to Google Drive: {ENV_PATH}')
    print('This will take ~5 minutes on first run. Subsequent runs will be instant.')

    # Install everything directly to the Drive path using --target
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'--target={ENV_PATH}', 'Cython'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'--target={ENV_PATH}', '--upgrade', 'numba'])

    print('Installing NeMo and WhisperX...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'--target={ENV_PATH}', 'nemo_toolkit[asr]'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'--target={ENV_PATH}',
                           'whisperx', 'google-auth', 'google-auth-oauthlib',
                           'google-auth-httplib2', 'google-api-python-client', 'pydub'])

    # Remove torch packages that pip pulled as dependencies —
    # Colab's pre-installed torch (with CUDA) must always be used instead.
    _cleanup_torch_from_target()

    print('\n=== Install to Drive complete. Restarting runtime to load new packages cleanly... ===')
    os.kill(os.getpid(), 9)

In [ ]:
# @title 2. Verify installation
import os

# WhisperX requires this env var on Colab to avoid pyannote weights_only errors
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'

import numpy as np
import pandas as pd

print('=== INSTALL SUMMARY ===')
print(f'numpy : {np.__version__}')
print(f'pandas: {pd.__version__}')

import nemo.collections.asr as nemo_asr
print('nemo.collections.asr: OK \u2713  (Parakeet path ready)')

try:
    import whisperx
    print(f'whisperx: OK \u2713  (alternative model \u2014 set USE_WHISPERX=True in config to use)')
except ImportError as e:
    print(f'whisperx: NOT AVAILABLE ({e})')
    print('  (Parakeet still works fine \u2014 only set USE_WHISPERX=True if you need it)')

print('=== All good ===')

In [ ]:
# @title 3. Authenticate Google Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
import io
import json
import os
from pathlib import Path
import time

auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Authenticated to your Google Drive.')

In [ ]:
# @title 4. CONFIGURATION - Paste your session folder ID here
# weblooper will tell you the exact value when you click the button.
# When launched via the weblooper 'Run in my Colab' button, the ID below is pre-filled automatically (no paste needed).
SESSION_FOLDER_ID = "__WEBLOOPER_SESSION_FOLDER_ID__"   # replaced by weblooper on Drive upload (or paste manually)

USE_PARAKEET = True   # Best for song lyrics + speed (recommended)
USE_WHISPERX = False  # Alternative for very accurate word-level timing
# Exactly one of the two above must be True. No silent fallbacks — choose deliberately.

print('Configuration ready.')

In [ ]:
# @title 5. Locate vocal stem in the Drive folder
def list_files_in_folder(folder_id):
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute()
    return results.get('files', [])

def download_file(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
    return dest_path

if not SESSION_FOLDER_ID:
    raise RuntimeError("Please paste your SESSION_FOLDER_ID above.")

files = list_files_in_folder(SESSION_FOLDER_ID)
vocal_file = next((f for f in files if 'vocal' in f['name'].lower() and f['name'].endswith(('.webm', '.mp3', '.wav'))), None)

if not vocal_file:
    raise RuntimeError("Could not find a vocals stem in the folder. Make sure the session is uploaded to Drive.")

work_dir = "/content/weblooper_lyrics"
os.makedirs(work_dir, exist_ok=True)
local_vocals = os.path.join(work_dir, vocal_file['name'])
download_file(vocal_file['id'], local_vocals)
print(f"Downloaded vocal stem: {local_vocals}")

OUTPUT_FOLDER_ID = SESSION_FOLDER_ID

In [ ]:
# @title 6. Run the AI model (Parakeet or WhisperX)
from pydub import AudioSegment

audio = AudioSegment.from_file(local_vocals)
duration_sec = len(audio) / 1000.0
print(f"Processing audio of length {duration_sec:.1f} seconds...")

# Convert to mono 16kHz WAV (models expect single-channel input;
# stereo vocals.webm causes shape mismatch errors in Parakeet)
local_vocals_mono = os.path.join(work_dir, 'vocals_mono.wav')
audio.set_channels(1).set_frame_rate(16000).export(local_vocals_mono, format='wav')
print(f"Converted to mono 16kHz WAV for model input.")

if USE_PARAKEET:
    print("Loading Parakeet TDT 0.6B (excellent for song lyrics)...")
    import nemo.collections.asr as nemo_asr
    model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
    result = model.transcribe([local_vocals_mono], timestamps=True)[0]
    text = result.text
    chunks = []
    if hasattr(result, 'timestamp') and result.timestamp:
        for w in result.timestamp.get('word', []):
            chunks.append({"text": w['word'], "timestamp": [w['start'], w['end']]})
    else:
        words = text.split()
        for i, w in enumerate(words):
            s = (i / max(1, len(words))) * duration_sec
            e = ((i + 1) / max(1, len(words))) * duration_sec
            chunks.append({"text": w, "timestamp": [s, e]})
    print("Parakeet finished.")

elif USE_WHISPERX:
    print("Loading WhisperX (best word-level timing)...")
    import whisperx
    device = "cuda"
    model = whisperx.load_model("large-v3", device, compute_type="float16")
    audio_wav = whisperx.load_audio(local_vocals_mono)
    result = model.transcribe(audio_wav, batch_size=16)
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio_wav, device)
    text = " ".join([seg["text"] for seg in result["segments"]])
    chunks = []
    for seg in result["segments"]:
        for word in seg.get("words", []):
            chunks.append({"text": word["word"], "timestamp": [word["start"], word["end"]]})
    print("WhisperX finished.")

else:
    raise RuntimeError("Set exactly one of USE_PARAKEET=True or USE_WHISPERX=True in the CONFIG cell.")

# Clean up the large temporary WAV file (can be 50-100MB+)
os.remove(local_vocals_mono)
print("Cleaned up temporary mono WAV.")

print("=== RAW TEXT (first 400 chars) ===")
print(text[:400] + "..." if len(text) > 400 else text)

In [ ]:
# @title 7. Build LyricTrack (weblooper format) — gap-based line splitting from word timestamps
import re

# --- Gap-based line splitting ---
# Song transcription models (Parakeet, WhisperX) often return text WITHOUT
# punctuation. Splitting on punctuation alone results in one giant block.
# Instead, we detect natural phrase boundaries by looking at silence gaps
# between consecutive words in the timestamp data.

GAP_THRESHOLD = 0.35   # seconds of silence between words to trigger a new line
MAX_WORDS_PER_LINE = 12  # safety cap even if no gap detected

has_real_timestamps = len(chunks) > 0 and chunks[0].get('timestamp', [0, 0])[1] > 0

segments = []

if has_real_timestamps:
    # Build lines by detecting gaps between words
    current_line_chunks = [chunks[0]]

    for i in range(1, len(chunks)):
        prev_end = chunks[i - 1]['timestamp'][1]
        curr_start = chunks[i]['timestamp'][0]
        gap = curr_start - prev_end

        # Start a new line if there's a significant gap or we hit the word cap
        if gap >= GAP_THRESHOLD or len(current_line_chunks) >= MAX_WORDS_PER_LINE:
            # Flush current line
            line_text = ' '.join(c['text'] for c in current_line_chunks)
            start = round(current_line_chunks[0]['timestamp'][0], 3)
            end = round(current_line_chunks[-1]['timestamp'][1], 3)
            segments.append({
                "id": f"colab_{int(time.time())}_{len(segments)}",
                "start": start,
                "end": end,
                "text": line_text,
                "source": "ai",
                "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
            })
            current_line_chunks = []

        current_line_chunks.append(chunks[i])

    # Flush the last line
    if current_line_chunks:
        line_text = ' '.join(c['text'] for c in current_line_chunks)
        start = round(current_line_chunks[0]['timestamp'][0], 3)
        end = round(current_line_chunks[-1]['timestamp'][1], 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{len(segments)}",
            "start": start,
            "end": end,
            "text": line_text,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })

    print(f"Built {len(segments)} segments using gap-based splitting (threshold={GAP_THRESHOLD}s, max_words={MAX_WORDS_PER_LINE}).")

else:
    # Fallback: uniform distribution (only if model returned no timestamps)
    def smart_split_lyrics(txt, max_chars=75):
        parts = re.split(r'([.!?\u3002\uff01\uff1f\n])', txt)
        lines = []
        current = ""
        for p in parts:
            current += p
            if len(current.strip()) > max_chars or p in '.!?\u3002\uff01\uff1f\n':
                if current.strip():
                    lines.append(current.strip())
                current = ""
        if current.strip():
            lines.append(current.strip())
        return [l for l in lines if l]

    lines = smart_split_lyrics(text)
    for i, line in enumerate(lines):
        start = round((i / max(1, len(lines))) * duration_sec, 3)
        end = round(((i + 1) / max(1, len(lines))) * duration_sec, 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{i}",
            "start": start,
            "end": end,
            "text": line,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })
    print(f"Built {len(segments)} segments using uniform timing (no word timestamps available).")

lyric_track = {
    "id": f"lt_colab_{int(duration_sec)}",
    "stemSessionId": SESSION_FOLDER_ID,
    "duration": duration_sec,
    "segments": segments,
    "metadata": {
        "generatedAt": int(time.time() * 1000),
        "lyricsModel": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3",
        "vocalsStemUsed": True,
        "source": "google-colab-free-gpu"
    },
    "version": 1
}

local_path = "/content/lyricTrack.json"
with open(local_path, "w") as f:
    json.dump(lyric_track, f, indent=2)

print(f"\nLyricTrack: {len(segments)} segments, duration {duration_sec:.1f}s")
print("=== TIMING PREVIEW (first 8 segments) ===")
for s in segments[:8]:
    print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s : {s['text'][:60]}")

In [ ]:
# @title 8. Fetch real lyrics from lyrics.ovh and correct segment text
import urllib.request
import urllib.parse

# ===========================================================================
# YouTube title parsing + smart query building
# ===========================================================================

# Common noise words found in YouTube music video titles
_NOISE_WORDS = [
    'official music video', 'official video', 'official audio', 'official lyric video',
    'lyric video', 'lyrics video', 'music video', 'with lyrics', 'w lyrics',
    'lyrics', 'lyric', 'audio', 'video',
    'official', 'officiel', 'oficial',
    'hd', 'hq', '4k', '1080p', '720p',
    'remastered', 'remaster', 'remasterizado',
    'live', 'en vivo', 'ao vivo', 'concert', 'tour',
    'full song', 'full album', 'full',
    'visualizer', 'visualiser', 'animated',
    'explicit', 'clean version', 'clean',
    'radio edit', 'single version', 'album version',
    'subtitulado', 'legendado', 'sub espanol', 'traduzione',
]

def _clean_title(raw):
    """Remove parenthesized/bracketed content and noise words from a title."""
    s = raw
    # Remove content in parentheses and brackets: (Official Video), [HD], (2017 Remaster)
    s = re.sub(r'\([^)]*\)', '', s)
    s = re.sub(r'\[[^\]]*\]', '', s)
    # Remove noise words (longest first to avoid partial matches)
    for word in sorted(_NOISE_WORDS, key=len, reverse=True):
        s = re.sub(r'\b' + re.escape(word) + r'\b', '', s, flags=re.IGNORECASE)
    # Remove trailing year like "2017" at the end
    s = re.sub(r'\b(19|20)\d{2}\b', '', s)
    # Collapse separators and extra whitespace
    s = re.sub(r'[|/]{1,2}|\u2014|\u2013|--|::', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # Remove trailing/leading punctuation junk
    s = s.strip(' -\u2014\u2013|/')
    return s

def _parse_artist_title(raw):
    """Try to split 'Artist - Song Title' from common YouTube patterns."""
    # Try common separators: " - ", " \u2014 ", " \u2013 ", " | "
    for sep in [' - ', ' \u2014 ', ' \u2013 ', ' | ', ' // ']:
        if sep in raw:
            parts = raw.split(sep, 1)
            artist = parts[0].strip()
            title = _clean_title(parts[1])
            if artist and title:
                return artist, title
    return None, None

def build_search_queries(raw_title):
    """Build a prioritized list of search queries from a YouTube video title."""
    queries = []
    
    # Tier 1: Try to parse "Artist - Title" directly
    artist, title = _parse_artist_title(raw_title)
    if artist and title:
        queries.append(f"{artist} {title}")
        queries.append(title)  # title alone as fallback
    
    # Tier 2: Full title cleaned of noise
    cleaned = _clean_title(raw_title)
    if cleaned:
        queries.append(cleaned)
    
    # Tier 3: Raw title as last resort (sometimes works if it's already clean)
    queries.append(raw_title.strip())
    
    # Deduplicate while preserving priority order
    seen = set()
    unique = []
    for q in queries:
        q_norm = q.lower().strip()
        if q_norm and q_norm not in seen:
            seen.add(q_norm)
            unique.append(q)
    return unique

def _lyrics_ovh_suggest(query):
    """Search lyrics.ovh suggest API. Returns list of results or []."""
    url = f"https://api.lyrics.ovh/suggest/{urllib.parse.quote(query, safe='')}"
    req = urllib.request.Request(url, headers={'User-Agent': 'weblooper-colab/1.0'})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode())
    return data.get('data', [])

def _lyrics_ovh_get(artist, title):
    """Fetch lyrics from the direct /v1 endpoint. Returns lyrics string or None."""
    url = f"https://api.lyrics.ovh/v1/{urllib.parse.quote(artist, safe='')}/{urllib.parse.quote(title, safe='')}"
    req = urllib.request.Request(url, headers={'User-Agent': 'weblooper-colab/1.0'})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode())
    return data.get('lyrics', '')

# ===========================================================================
# Main lyrics lookup logic
# ===========================================================================

# --- Extract song title from meta.json in the Drive folder ---
meta_file_entry = next((f for f in files if f['name'] == 'meta.json'), None)
raw_song_title = None

if meta_file_entry:
    meta_path = os.path.join(work_dir, 'meta.json')
    download_file(meta_file_entry['id'], meta_path)
    with open(meta_path) as f:
        session_meta = json.load(f)
    # Try youtubeVideoTitle first, then fileName
    raw_song_title = session_meta.get('youtubeVideoTitle', '')
    if not raw_song_title:
        fn = session_meta.get('fileName', '')
        # Strip prefixes like "YouTube \u2014 " and file extensions
        raw_song_title = re.sub(r'^youtube\s*[\u2014\u2013-]\s*', '', fn, flags=re.IGNORECASE)
        raw_song_title = re.sub(r'\.(mp3|wav|webm|ogg|m4a)$', '', raw_song_title, flags=re.IGNORECASE)
    raw_song_title = raw_song_title.strip()

real_lyrics = None
artist_name = None
song_title = None

if not raw_song_title:
    print("Could not determine song title from meta.json. Skipping lyrics lookup.")
    print("Segments will use AI-transcribed text as-is.")
else:
    print(f"Raw title from session: {raw_song_title}")

    # --- Tier 1: Direct /v1 lookup if we can parse Artist - Title ---
    parsed_artist, parsed_title = _parse_artist_title(raw_song_title)
    if parsed_artist and parsed_title:
        print(f"  Tier 1: Trying direct lookup: {parsed_artist} / {parsed_title}")
        try:
            direct_lyrics = _lyrics_ovh_get(parsed_artist, parsed_title)
            if direct_lyrics and direct_lyrics.strip():
                real_lyrics = direct_lyrics
                artist_name = parsed_artist
                song_title = parsed_title
                print(f"  \u2713 Direct lookup succeeded!")
        except Exception as e:
            print(f"  Tier 1 failed: {e}")

    # --- Tier 2: Progressive suggest search with cleaned queries ---
    if not real_lyrics:
        queries = build_search_queries(raw_song_title)
        print(f"  Tier 2: Trying suggest search with {len(queries)} queries: {queries}")

        for i, query in enumerate(queries):
            try:
                results = _lyrics_ovh_suggest(query)
                if not results:
                    print(f"    Query {i+1} '{query}': no results")
                    continue

                # Pick best result: among those within 10s of our duration, prefer highest rank (most popular)
                best = results[0]
                if duration_sec > 0:
                    candidates = [r for r in results[:5] if abs(r.get('duration', 0) - duration_sec) < 10]
                    if candidates:
                        best = max(candidates, key=lambda r: r.get('rank', 0))
                    else:
                        # No close duration match, fall back to closest duration
                        best = min(results[:5], key=lambda r: abs(r.get('duration', 0) - duration_sec))

                artist_name = best['artist']['name']
                song_title = best['title']
                match_dur = best.get('duration', 0)
                print(f"    Query {i+1} '{query}': found {artist_name} \u2014 {song_title} ({match_dur}s vs our {duration_sec:.0f}s)")

                # Fetch the actual lyrics
                fetched = _lyrics_ovh_get(artist_name, song_title)
                if fetched and fetched.strip():
                    real_lyrics = fetched
                    print(f"  \u2713 Got lyrics via suggest search!")
                    break
                else:
                    print(f"    (lyrics endpoint returned empty for this match)")
            except Exception as e:
                print(f"    Query {i+1} '{query}': error - {e}")
                continue

    # --- Apply lyrics correction if we found real lyrics ---
    if real_lyrics and real_lyrics.strip():
        # Split into lines, filter empty lines and bracketed annotations like [guitar solo]
        real_lines = [l.strip() for l in real_lyrics.split('\n')
                      if l.strip() and not l.strip().startswith('[')]
        print(f"\nReplacing AI segments with {len(real_lines)} real lyric lines from lyrics.ovh...")
        print(f"(Using AI output only for timestamps, text comes entirely from lyrics.ovh)")

        # Trigram similarity \u2014 used only to find which AI segment best matches
        # each real line, so we can steal its start timestamp.
        def _normalize(s):
            return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', '', s.lower())).strip()

        def _trigram_similarity(a, b):
            na, nb = _normalize(a), _normalize(b)
            if na == nb:
                return 1.0
            if not na or not nb:
                return 0.0
            ta = set(na[i:i+3] for i in range(len(na) - 2))
            tb = set(nb[i:i+3] for i in range(len(nb) - 2))
            if not ta or not tb:
                return 0.0
            intersection = len(ta & tb)
            return (2 * intersection) / (len(ta) + len(tb))

        # === ALIGNMENT: High-confidence anchors + proportional positional fill ===
        #
        # Strategy (simple and robust):
        # 1. Find 'anchor' lines where AI text clearly matches real lyrics (score >= 0.7)
        #    These pin the alignment at reliable points.
        # 2. For gaps between anchors: DON'T try text matching (AI text is often garbled).
        #    Instead, take the AI segments in that time range and assign real lines
        #    proportionally by position. The AI timestamps are accurate even when the
        #    transcribed words are wrong.
        # 3. Any remaining unassigned lines get linearly interpolated.

        ANCHOR_THRESHOLD = 0.65  # high confidence only
        SEARCH_WINDOW = 12      # segments ahead to search for anchors

        def _best_match_in_range(text, segs, start, end):
            """Find best trigram match for text among segments[start:end].
            Tries single segments and concatenations of 2-3."""
            best_score = 0.0
            best_idx = -1
            for si in range(start, min(end, len(segs))):
                score = _trigram_similarity(text, segs[si]['text'])
                if score > best_score:
                    best_score = score
                    best_idx = si
                if si + 1 < len(segs):
                    c2 = segs[si]['text'] + ' ' + segs[si + 1]['text']
                    s2 = _trigram_similarity(text, c2)
                    if s2 > best_score:
                        best_score = s2
                        best_idx = si
                if si + 2 < len(segs):
                    c3 = segs[si]['text'] + ' ' + segs[si + 1]['text'] + ' ' + segs[si + 2]['text']
                    s3 = _trigram_similarity(text, c3)
                    if s3 > best_score:
                        best_score = s3
                        best_idx = si
            return best_score, best_idx

        # --- Step 1: Find high-confidence anchors ---
        # anchors: real_line_index -> (timestamp, ai_segment_index)
        anchors = {}
        ai_ptr = 0

        for i, real_line in enumerate(real_lines):
            score, seg_idx = _best_match_in_range(real_line, segments, ai_ptr, ai_ptr + SEARCH_WINDOW)
            if score >= ANCHOR_THRESHOLD and seg_idx >= 0:
                anchors[i] = (segments[seg_idx]['start'], seg_idx)
                ai_ptr = seg_idx + 1

        print(f"  Step 1: Found {len(anchors)} high-confidence anchors out of {len(real_lines)} lines.")

        # --- Step 2: Validate anchors (single-line jumps > 15s are always wrong) ---
        sorted_anchor_keys = sorted(anchors.keys())
        demoted = set()
        for k in range(len(sorted_anchor_keys) - 1):
            idx_a = sorted_anchor_keys[k]
            idx_b = sorted_anchor_keys[k + 1]
            time_a = anchors[idx_a][0]
            time_b = anchors[idx_b][0]
            line_gap = idx_b - idx_a
            time_gap = time_b - time_a
            # A single line can never span > 15 seconds
            if line_gap == 1 and time_gap > 15.0:
                demoted.add(idx_b)
            # Multiple lines: if time_gap > line_gap * 12s, something is wrong
            elif line_gap > 1 and time_gap > line_gap * 12.0:
                demoted.add(idx_b)

        if demoted:
            print(f"  Step 2: Demoted {len(demoted)} anchors with impossible time jumps.")
            for d in demoted:
                del anchors[d]

        # --- Step 3: Positional fill for gaps between anchors ---
        # For each gap, assign unanchored lines to AI segments proportionally.
        sorted_anchor_keys = sorted(anchors.keys())

        # Add virtual boundary anchors if needed
        first_time = segments[0]['start'] if segments else 0.0
        last_time = segments[-1]['end'] if (segments and segments[-1].get('end')) else (segments[-1]['start'] + 3.0 if segments else 0.0)

        # Build full anchor list including virtual boundaries
        all_anchors = {}  # line_index -> (time, seg_idx)
        all_anchors.update(anchors)
        if 0 not in all_anchors:
            all_anchors[-1] = (first_time, 0)  # virtual start
        if (len(real_lines) - 1) not in all_anchors:
            all_anchors[len(real_lines)] = (last_time, len(segments) - 1)  # virtual end

        sorted_all = sorted(all_anchors.keys())
        positional_count = 0

        for k in range(len(sorted_all) - 1):
            prev_idx = sorted_all[k]
            next_idx = sorted_all[k + 1]

            gap_lines = [i for i in range(prev_idx + 1, next_idx) if i not in all_anchors]
            if not gap_lines:
                continue

            # Get AI segments in this time range
            prev_seg = all_anchors[prev_idx][1]
            next_seg = all_anchors[next_idx][1]
            ai_segs_in_gap = segments[prev_seg + 1 : next_seg]

            n_lines = len(gap_lines)
            n_segs = len(ai_segs_in_gap)

            if n_segs > 0:
                # Proportional assignment: distribute lines across AI segments
                for i, line_idx in enumerate(gap_lines):
                    # Map line position to segment position proportionally
                    seg_pos = int(i * n_segs / n_lines)
                    seg_pos = min(seg_pos, n_segs - 1)
                    seg_data = ai_segs_in_gap[seg_pos]
                    seg_global_idx = prev_seg + 1 + seg_pos
                    all_anchors[line_idx] = (seg_data['start'], seg_global_idx)
                    positional_count += 1
            # else: no AI segments in gap, will be interpolated in Step 4

        if positional_count:
            print(f"  Step 3: Assigned {positional_count} lines by positional mapping to AI timestamps.")

        # --- Step 4: Build final segments with interpolation for any remaining ---
        anchor_times = {k: v[0] for k, v in all_anchors.items()}
        sorted_time_keys = sorted(anchor_times.keys())

        new_segments = []
        for i, real_line in enumerate(real_lines):
            if i in anchor_times:
                start_time = anchor_times[i]
            else:
                # Linear interpolation between nearest assigned neighbors
                prev_k = max(k for k in sorted_time_keys if k < i)
                next_k = min(k for k in sorted_time_keys if k > i)
                prev_time = anchor_times[prev_k]
                next_time = anchor_times[next_k]
                span = next_k - prev_k
                position = i - prev_k
                start_time = prev_time + (next_time - prev_time) * position / span

            new_segments.append({
                'text': real_line,
                'start': round(start_time, 2),
                'end': None,
                'source': 'lyrics.ovh'
            })

        # Fill end times
        for i in range(len(new_segments) - 1):
            new_segments[i]['end'] = new_segments[i + 1]['start']
        if new_segments:
            new_segments[-1]['end'] = round(last_time, 2)

        # Ensure monotonically increasing timestamps (safety net)
        for i in range(1, len(new_segments)):
            if new_segments[i]['start'] < new_segments[i - 1]['start']:
                new_segments[i]['start'] = new_segments[i - 1]['start'] + 0.1
            if new_segments[i - 1]['end'] != new_segments[i]['start']:
                new_segments[i - 1]['end'] = new_segments[i]['start']

        # Remove virtual boundary keys for reporting
        real_anchor_count = len([k for k in anchors.keys() if k >= 0])
        interpolated_count = len(real_lines) - real_anchor_count - positional_count
        print(f"  Step 4: Built {len(new_segments)} segments. {real_anchor_count} anchored, {positional_count} positional, {max(0, interpolated_count)} interpolated.")

        print(f"\n=== PREVIEW (segments 35-48) ===")
        for idx, s in enumerate(new_segments[35:48], start=35):
            tag = ' [A]' if idx in anchors else (' [P]' if idx in all_anchors and idx not in anchors else '')
            print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s : {s['text'][:50]}{tag}")

        # Replace segments entirely
        segments = new_segments
        lyric_track['segments'] = segments
        lyric_track['metadata']['lyricsSource'] = f"lyrics.ovh ({artist_name} - {song_title})"

        # Re-write the local file with new segments
        with open(local_path, "w") as f:
            json.dump(lyric_track, f, indent=2)

        print(f"\n=== PREVIEW (first 12 segments) ===")
        for s in segments[:12]:
            tag = ' *' if s['text'] in [real_lines[k] for k in anchor_times] else ''
            print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s : {s['text'][:55]}{tag}")
    else:
        if raw_song_title:
            print("\nCould not find lyrics on lyrics.ovh. Keeping AI-transcribed text.")
            print("You can manually correct later in weblooper (Use my own lyrics button).")

In [ ]:
# @title 9. Write back to your Drive folder (lyricTrack.json + patch meta.json)
from googleapiclient.http import MediaIoBaseUpload
from datetime import datetime, timezone

# Stamp the lyricTrack with processing time so weblooper can distinguish
# fresh results from stale ones (poller ignores tracks older than session start).
lyric_track['metadata']['processedAt'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

if OUTPUT_FOLDER_ID:
    # Delete any previous lyricTrack.json to avoid duplicates
    try:
        old_files = drive_service.files().list(
            q=f"'{OUTPUT_FOLDER_ID}' in parents and name='lyricTrack.json' and trashed=false",
            fields="files(id)"
        ).execute().get('files', [])
        for old in old_files:
            drive_service.files().delete(fileId=old['id']).execute()
        if old_files:
            print(f"Deleted {len(old_files)} previous lyricTrack.json file(s).")
    except Exception as e:
        print(f"Warning: could not clean up old files: {e}")

    # Upload fresh lyricTrack.json
    media = MediaIoBaseUpload(open(local_path, "rb"), mimetype="application/json")
    drive_service.files().create(
        body={"name": "lyricTrack.json", "parents": [OUTPUT_FOLDER_ID]},
        media_body=media
    ).execute()
    print("Uploaded lyricTrack.json")

    # Patch meta.json so weblooper picks it up automatically
    try:
        meta_search = drive_service.files().list(
            q=f"'{OUTPUT_FOLDER_ID}' in parents and name='meta.json' and trashed=false",
            fields="files(id)"
        ).execute().get('files', [])
        if meta_search:
            meta_id = meta_search[0]['id']
            req = drive_service.files().get_media(fileId=meta_id)
            meta_bytes = io.BytesIO()
            downloader = MediaIoBaseDownload(meta_bytes, req)
            done = False
            while not done:
                _, done = downloader.next_chunk()
            current_meta = json.loads(meta_bytes.getvalue().decode('utf-8'))
            current_meta['lyricTrack'] = lyric_track
            media = MediaIoBaseUpload(
                io.BytesIO(json.dumps(current_meta, indent=2).encode('utf-8')),
                mimetype='application/json'
            )
            drive_service.files().update(fileId=meta_id, media_body=media).execute()
            print("Patched meta.json with lyricTrack \u2014 weblooper will see it on reload!")
    except Exception as e:
        print(f"Could not patch meta.json: {e}")

    print(f"\n=== SUCCESS (processedAt: {lyric_track['metadata']['processedAt']}) ===")
    print("Go back to weblooper \u2014 it should detect the results automatically within 15 seconds.")
else:
    print("No folder ID \u2014 please download the file manually and place it in your session folder.")
    from google.colab import files
    files.download(local_path)


**That's it!** 

The results are written back to the exact same Drive folder weblooper uses. 
Reload the session in the app and the timed lyrics will appear.